# Simulación: Recta de Altura (Método de Marcq St. Hilaire)

El método de **Marcq St. Hilaire** es el pilar de la navegación astronómica moderna. Consiste en comparar la Altura Verdadera ($a_v$) de un astro obtenida con el sextante, con la Altura Estimada ($a_e$) calculada matemáticamente asumiendo que estamos en nuestra Situación de Estima ($l_e, L_e$).

La diferencia entre ambas ($\Delta a = a_v - a_e$) nos indica a cuántas millas de distancia de nuestra estima debemos trazar una línea perpendicular al Azimut del astro para obtener nuestra Recta de Altura (LOP).

En este laboratorio resolveremos el Triángulo de Posición esférico paso a paso.

In [ ]:
import math

def grados_a_rad(grados, minutos, direccion):
    """Convierte coordenadas (Grados, Minutos, Dirección) a Radianes, con el signo correcto."""
    decimal = grados + (minutos / 60.0)
    # Norte / Este son positivos. Sur / Oeste son negativos.
    if direccion.upper() in ['S', 'W', 'O']:
        decimal = -decimal
    return math.radians(decimal)

def rad_a_grados_minutos(rad):
    """Convierte radianes de vuelta a Grados y Minutos absolutos."""
    decimal = math.degrees(rad)
    grados = int(abs(decimal))
    minutos = (abs(decimal) - grados) * 60.0
    return grados, minutos

print("✅ Funciones de conversión trigonométrica cargadas.")

## 1. Introducción de Datos (Situación de Estima y Observación)
Ingresa aquí los datos de tu estima y los datos que has extraído del Almanaque Náutico.

In [ ]:
# --- DATOS DE ENTRADA ---

# Situación de Estima
l_e_grados = 36  # Latitud
l_e_minutos = 0.0
l_e_dir = 'N'    # 'N' o 'S'

# Datos del Almanaque para la hora de la observación
dec_grados = 15  # Declinación del astro
dec_minutos = 25.5
dec_dir = 'N'    # 'N' o 'S'

P_grados = 45    # Ángulo en el Polo (calculado previamente a partir del hl)
P_minutos = 30.0
P_dir = 'W'      # 'E' o 'W'

# Altura Verdadera observada con el sextante (corregida)
a_v_grados = 42
a_v_minutos = 10.5

print("✅ Datos de entrada registrados.")

## 2. Cálculo de la Altura Estimada ($a_e$)
Aplicaremos la fórmula fundamental de la trigonometría esférica (Teorema de los Cosenos para los lados) aplicada al Triángulo Astronómico:
$\sin(a_e) = \sin(l_e) \cdot \sin(d) + \cos(l_e) \cdot \cos(d) \cdot \cos(P)$

In [ ]:
# Conversión a radianes
l_e_rad = grados_a_rad(l_e_grados, l_e_minutos, l_e_dir)
d_rad = grados_a_rad(dec_grados, dec_minutos, dec_dir)
P_rad = grados_a_rad(P_grados, P_minutos, P_dir)

# Fórmula de la Altura Estimada
sin_ae = (math.sin(l_e_rad) * math.sin(d_rad)) + (math.cos(l_e_rad) * math.cos(d_rad) * math.cos(P_rad))
a_e_rad = math.asin(sin_ae)

ae_g, ae_m = rad_a_grados_minutos(a_e_rad)

print(f"Altura Estimada calculada (ae): {ae_g}º {ae_m:.2f}'")

## 3. Cálculo del Azimut Estimado ($Z_e$)
Se obtiene mediante el teorema de los cosenos o fórmulas análogas de las cotangentes. Calcularemos primero el ángulo del Azimut Náutico (desde el polo elevado) y lo convertiremos a cuadrantal/circular.

In [ ]:
# Numerador y Denominador para el cálculo de cotangente del Azimut
# cot(Z) = [tan(d) * cos(l_e) / sin(P)] - [sin(l_e) / tan(P)]
numerador = math.sin(d_rad) - (math.sin(l_e_rad) * sin_ae)
denominador = math.cos(l_e_rad) * math.cos(a_e_rad)
cos_Z = numerador / denominador

# Evitar errores de redondeo de punto flotante
cos_Z = max(-1.0, min(1.0, cos_Z))
Z_rad = math.acos(cos_Z)
Z_g, Z_m = rad_a_grados_minutos(Z_rad)

# Determinar el cuadrante del Azimut
# El Azimut toma el nombre del polo elevado (Latitud) y hacia el Este/Oeste del Ángulo en el Polo (P)
polo = 'N' if l_e_grados >= 0 else 'S'
rumbo_p = 'W' if P_dir == 'W' else 'E'

print(f"Azimut Cuadrantal (Ze): {polo} {Z_g}º {Z_m:.1f}' {rumbo_p}")

# Convertir a Circular (000º - 360º)
z_decimal = Z_g + (Z_m/60.0)
if polo == 'N' and rumbo_p == 'E': Z_circular = z_decimal
elif polo == 'N' and rumbo_p == 'W': Z_circular = 360 - z_decimal
elif polo == 'S' and rumbo_p == 'E': Z_circular = 180 - z_decimal
else: Z_circular = 180 + z_decimal  # S y W

print(f"Azimut Circular (Ze): {Z_circular:.1f}º")

## 4. El Determinante (Diferencia de Alturas)
$\Delta a = a_v - a_e$
Si es positivo (+), la Recta de Altura se acerca al astro desde la estima.
Si es negativo (-), la Recta de Altura se aleja del astro desde la estima.

In [ ]:
av_decimal = a_v_grados + (a_v_minutos / 60.0)
ae_decimal = ae_g + (ae_m / 60.0)

delta_a_minutos = (av_decimal - ae_decimal) * 60.0

print("=== RESULTADO DEL DETERMINANTE ===")
print(f"Diferencia de Alturas (Δa): {delta_a_minutos:+.1f} millas náuticas")
if delta_a_minutos > 0:
    print("Significado: Trazar hacia el astro (+)")
elif delta_a_minutos < 0:
    print("Significado: Trazar en contra del astro (-)")
else:
    print("Significado: La Recta pasa exactamente por la Situación de Estima.")